<a href="https://colab.research.google.com/github/Linachvn/Chatbot-Financier/blob/main/P5_Part1_GetData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import os
import re
from bs4 import BeautifulSoup
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks/projet5')

Mounted at /content/drive


In [ ]:
companies ={
'company_names' : ['IBM', 'Apple', 'Microsoft', 'Amazon', 'Tesla', 'Uber', 'Meta', 'Google', 'Nvidia', 'Oracle'],
'companies_alpha' : ['IBM', 'AAPL', 'MSFT', 'AMZN', 'TSLA', 'UBER', 'META', 'GOOG', 'NVDA', 'ORCL'],
'companies_wiki': ['IBM', 'Apple_Inc.', 'Microsoft', 'Amazon_(company)', 'Tesla,_Inc.', 'Uber', 'Meta_Platforms', 'Google', 'Nvidia', 'Oracle_Corporation']
}

df = pd.DataFrame(companies)
print(df)

  company_names companies_alpha      companies_wiki
0           IBM             IBM                 IBM
1         Apple            AAPL          Apple_Inc.
2     Microsoft            MSFT           Microsoft
3        Amazon            AMZN    Amazon_(company)
4         Tesla            TSLA         Tesla,_Inc.
5          Uber            UBER                Uber
6          Meta            META      Meta_Platforms
7        Google            GOOG              Google
8        Nvidia            NVDA              Nvidia
9        Oracle            ORCL  Oracle_Corporation


# Getting news for each company

In [ ]:
api_key="HJDLBBLWFM4MJPIQ"
for i in range(len(companies['companies_alpha'])):
    company = companies['companies_alpha'][i]
    company_name = companies['company_names'][i]

    url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={company}&apikey={api_key}"
    response = requests.get(url)
    data = response.json()

    # Create a text file for each company
    with open(f"data/{company_name}_news.txt", "w", encoding="utf-8") as file:
        # Check if the data contains 'feed' which usually holds the news articles
        if 'feed' in data:
            for article in data['feed']:
                # Extract title and summary if they exist
                title = article.get('title', 'No title available')
                summary = article.get('summary', 'No summary available')
                print(summary)
                # Write title and summary to the file
                file.write(summary)
        else:
            file.write(f"No news data available for {company_name}\n")

    print(f"News data for {company_name} has been saved to {company_name}_news.txt")

With PLTR's notable rise in stock price, it is crucial to assess its current position and decide if it continues to be a promising investment opportunity.
Scott Helfstein emphasizes U.S. competitiveness themes like infrastructure, defense, and automation for resilient portfolios in 2025. Nuclear energy and AI-driven automation are key drivers of growth amid rising geopolitical and economic challenges.
QBTS stock benefits from an expanding clientele and its growing influence in the quantum computing space.
International Business Machines forms a collaboration with defense and security intelligence provider Janes to support AI integration in defense applications.
ADBE's fourth-quarter fiscal 2024 results are likely to reflect strong demand for Generative AI solutions amid growing competition and stretched valuation.
TD SYNNEX partners with LastPass to make the latter's password and identity management products accessible to IT resellers.
Many companies dealing in artificial intelligence 

# Getting Balance Sheet for each company

In [ ]:
api_key="HJDLBBLWFM4MJPIQ"
# Function to convert camel case keys to readable descriptions
def camel_case_to_readable(text):
    return re.sub(r'(?<!^)(?=[A-Z])', ' ', text)

# Loop through each company
for i in range(len(companies['companies_alpha'])):
    company = companies['companies_alpha'][i]
    company_name = companies['company_names'][i]
    # Fetch the balance sheet data
    url = f'https://www.alphavantage.co/query?function=BALANCE_SHEET&symbol={company}&apikey={api_key}'
    response = requests.get(url)
    data = response.json()
    print(data)

    # Open a text file for writing the balance sheet data for this company
    with open(f'data/{company_name}_balance_sheet.txt', 'w') as file:
        file.write(f"Balance Sheet Data for {company_name}\n")
        file.write("\n\n")

        # Process annual reports
        if "annualReports" in data:
            file.write("Annual Reports:\n\n")
            for report in data["annualReports"]:
                report_text = []
                year = "unknown"

                for key, value in report.items():
                    description = camel_case_to_readable(key)
                    formatted_value = f"{int(value):,}" if value.isdigit() else value
                    report_text.append(f"Annual Report of {company_name} for the year {year} states that the {description.lower()} is {formatted_value}.")

                    # Extract the fiscal year if available
                    if description.lower() == 'fiscal date ending':
                        year = formatted_value[0:4]

                full_text = " ".join(report_text)
                file.write(full_text)

        # Process quarterly reports
        if "quarterlyReports" in data:
            file.write("Quarterly Reports:\n\n")
            for report in data["quarterlyReports"]:
                report_text = []
                year = "unknown"
                quarter = "unknown quarter"

                for key, value in report.items():
                    description = camel_case_to_readable(key)
                    formatted_value = f"{int(value):,}" if value.isdigit() else value
                    report_text.append(f"Quarterly Report of {company_name} for the {quarter} of the year {year} states that the {description.lower()} is {formatted_value}.")

                    # Extract year and quarter if available
                    if description.lower() == 'fiscal date ending':
                        year = formatted_value[0:4]
                        month = formatted_value[5:7]
                        if month == '03':
                            quarter = 'first quarter'
                        elif month == '06':
                            quarter = 'second quarter'
                        elif month == '09':
                            quarter = 'third quarter'
                        elif month == '12':
                            quarter = 'fourth quarter'

                full_text = " ".join(report_text)
                file.write(full_text)

        # Separator for end of report

print("Data has been written to text files for each company.")


{'symbol': 'IBM', 'annualReports': [{'fiscalDateEnding': '2023-12-31', 'reportedCurrency': 'USD', 'totalAssets': '135241000000', 'totalCurrentAssets': '32908000000', 'cashAndCashEquivalentsAtCarryingValue': '13068000000', 'cashAndShortTermInvestments': '13068000000', 'inventory': '1161000000', 'currentNetReceivables': '7725000000', 'totalNonCurrentAssets': '101302000000', 'propertyPlantEquipment': '-472000000', 'accumulatedDepreciationAmortizationPPE': 'None', 'intangibleAssets': '71214000000', 'intangibleAssetsExcludingGoodwill': '11036000000', 'goodwill': '60178000000', 'investments': '125000000', 'longTermInvestments': '125000000', 'shortTermInvestments': '373000000', 'otherCurrentAssets': '10587000000', 'otherNonCurrentAssets': 'None', 'totalLiabilities': '112628000000', 'totalCurrentLiabilities': '34122000000', 'currentAccountsPayable': '4132000000', 'deferredRevenue': '16984000000', 'currentDebt': '12851000000', 'shortTermDebt': '6426000000', 'totalNonCurrentLiabilities': '870720

# Getting Income statement for each company



In [ ]:
for i in range(len(companies['companies_alpha'])):
    company = companies['companies_alpha'][i]
    company_name = companies['company_names'][i]
    # Fetch the balance sheet data
    url = f'https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={company}&apikey={api_key}'
    response = requests.get(url)
    data = response.json()

    # Open a text file for writing the balance sheet data for this company
    with open(f'data/{company_name}_income_statement.txt', 'w') as file:
        file.write(f"Income Statement Data for {company_name}\n")
        file.write("\n\n")

        # Process annual reports
        if "annualReports" in data:
            file.write("Annual Reports:\n\n")
            for report in data["annualReports"]:
                report_text = []
                year = "unknown"

                for key, value in report.items():
                    description = camel_case_to_readable(key)
                    formatted_value = f"{int(value):,}" if value.isdigit() else value
                    report_text.append(f"Annual Report of {company_name} for the year {year} states that the {description.lower()} is {formatted_value}.")

                    # Extract the fiscal year if available
                    if description.lower() == 'fiscal date ending':
                        year = formatted_value[0:4]

                full_text = " ".join(report_text)
                file.write(full_text)

        # Process quarterly reports
        if "quarterlyReports" in data:
            file.write("Quarterly Reports:\n\n")
            for report in data["quarterlyReports"]:
                report_text = []
                year = "unknown"
                quarter = "unknown quarter"

                for key, value in report.items():
                    description = camel_case_to_readable(key)
                    formatted_value = f"{int(value):,}" if value.isdigit() else value
                    # Extract year and quarter if available
                    if description.lower() == 'fiscal date ending':
                        year = formatted_value[0:4]
                        month = formatted_value[5:7]
                        if month == '03':
                            quarter = 'first quarter'
                        elif month == '06':
                            quarter = 'second quarter'
                        elif month == '09':
                            quarter = 'third quarter'
                        elif month == '12':
                            quarter = 'fourth quarter'
                    report_text.append(f"Quarterly Report of {company_name} for the {quarter} of the year {year} states that the {description.lower()} is {formatted_value}.")

# getting wikipedia page for each company

In [ ]:
for i in range(len(companies['companies_wiki'])):
    company = companies['companies_wiki'][i]
    company_name = companies['company_names'][i]
    # Send a request to fetch the content of the page
    response = requests.get(f"https://en.wikipedia.org/wiki/{company}")

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find the main content area where the article text is located
        content_div = soup.find('div', {'class': 'mw-content-ltr mw-parser-output'})

        # Initialize an empty string to collect all text
        page_text = ""

        if content_div:
            # Iterate over all paragraph tags within the main content div
            for paragraph in content_div.find_all(['p', 'h2', 'h3']):
                page_text += paragraph.get_text(separator=' ', strip=True) + "\n\n"

            # Save the extracted text to a file
            with open(f"data/{company_name}_Wikipedia.txt", "w", encoding="utf-8") as file:
                file.write(page_text)

            print(f"Page content has been saved to '{company_name}_Wikipedia.txt'")
        else:
            print("Main content area not found.")
    else:
        print("Failed to retrieve the page. Status code:", response.status_code)

Page content has been saved to 'IBM_Wikipedia.txt'
Page content has been saved to 'Apple_Wikipedia.txt'
Page content has been saved to 'Microsoft_Wikipedia.txt'
Page content has been saved to 'Amazon_Wikipedia.txt'
Page content has been saved to 'Tesla_Wikipedia.txt'
Page content has been saved to 'Uber_Wikipedia.txt'
Page content has been saved to 'Meta_Wikipedia.txt'
Page content has been saved to 'Google_Wikipedia.txt'
Page content has been saved to 'Nvidia_Wikipedia.txt'
Page content has been saved to 'Oracle_Wikipedia.txt'
